# Load ChromaDB

In [ ]:
# Mount Google Drive to access and save files
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import chromda db
try:
  import chromadb
except:
  !pip install chromadb
  import chromadb

In [ ]:
# Set up client and set path to google drive
client = chromadb.PersistentClient(path="/content/drive/MyDrive/chroma_db")

# Initialize movie chunk collection and count entries
collection = client.get_collection(name="movie_chunks")
collection.count()

# Query Embedding Function

In [ ]:
# Import sentence transformers
try:
  from sentence_transformers import SentenceTransformer
except:
  !pip install sentence_transformers
  from sentence_transformers import SentenceTransformer

In [ ]:
# Load pretrained model
embedding_model = SentenceTransformer("all-mpnet-base-v2")

In [ ]:
def get_query_embedding(query, embedding_model):
  """
  This function takes a input query removes leading and
  lagging whitespaces, then generates an embedding for the query.

  The resulting embedding is converted to a Python list format, making it
  compatible with vector databases such as Chroma.

  Parameters
  ----------
  query: str
    The input query string to be embedded

  Returns
  --------
  embedding_model : object
        A preloaded SentenceTransformer model
        (in this case all-mpnet-base-v2).
  """
  return embedding_model.encode(query.strip()).tolist()

# Retrieval Function

In [ ]:
def retrieve_documents(query, collection, embedding_model, top_k=5):
    """
    This function takes an input query, generates its embedding,
    and uses that embedding to retrieve the top-k most relevant
    document chunks from a Chroma collection.

    It returns both the retrieved document texts and their
    corresponding chunk IDs, which can be used for source
    referencing in a RAG prompt.

    Parameters
    ----------
    query : str
        The input query string used for retrieval

    collection : object
        The Chroma collection containing embedded document chunks

    model : object
        A preloaded SentenceTransformer model used to generate the query embedding
    top_k : int, optional
        The number of top matching document chunks to retrieve, the default is 5

    Returns
    -------
    tuple
        A tuple containing:
          documents : list
              A list of the retrieved documents
          ids : list
              A corresponding list of the chunk IDs for the retrieved documents
    """
    # Call query embedding function
    query_embedding = get_query_embedding(query, embedding_model)

    # Query chroma db
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents"]
    )
    # Return docs and ids
    return results["documents"][0], results["ids"][0]

## Test Cases

In [ ]:
# Define function for printing outputs from test cases
def print_test_output(documents, ids):
  """
  Doc-string here....
  """
  # View each result
  for i in range(len(documents)):

      # Shorten plots
      preview = documents[i][:300]
      if len(documents[i]) > 300:
          preview += "..."

      # Print result
      print(f"Result {i+1}")
      print(f"ID: {ids[i]}")
      print(f"{preview}")
      print("-" * 80)

### Title-Based

#### 1. What is the plot of The Hunger Games?

In [ ]:
query1 = "What is the plot of The Hunger Games?"
documents, ids = retrieve_documents(query1, collection, embedding_model, top_k=3)

In [ ]:
# View each result
print_test_output(documents, ids)

#### 2. What happens in the Titanic?

In [ ]:
query2 = "What happens in the Titanic?"
documents, ids = retrieve_documents(query2, collection, embedding_model, top_k=3)
print_test_output(documents, ids)


#### 3. Give me the story of Star Wars Episode V: The Empire Strikes Back.

In [ ]:
query3 = "Give me the story of Star Wars Episode V: The Empire Strikes Back."
documents, ids = retrieve_documents(query3, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

#### 4. What is the plot of Sex and the City the Movie?

In [ ]:
query4 = "What is the plot of Sex and the City the Movie?"
documents, ids = retrieve_documents(query4, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

#### 5. What is the plot of Batman Begins?

In [ ]:
query5 = "What is the plot of Batman Begins?"
documents, ids = retrieve_documents(query5, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

### Character-Based

#### 6. What happens between Carrie and Big in Sex and The City the Movie?

In [ ]:
query6 = "What happens between Carrie and Big in Sex and The City the Movie?"
documents, ids = retrieve_documents(query6, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

#### 7. What happens between Jack and Rose in Titanic?

In [ ]:
query7 = "What happens between Jack and Rose in Titanic? "
documents, ids = retrieve_documents(query7, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

#### 8. Who is Katniss and what role does she play in The Hunger Games?

In [ ]:
query8 = "Who is Katniss and what role does she play in The Hunger Games?"
documents, ids = retrieve_documents(query8, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

#### 9. What is the relationship between Luke and Darth Vader in Star Wars Episode V?

In [ ]:
query9 = "What is the relationship between Luke and Darth Vader in Star Wars Episode V?"
documents, ids = retrieve_documents(query9, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

#### 10. Tell me about Darth Vader

In [ ]:
query10 = "Tell me about Darth Vader"
documents, ids = retrieve_documents(query10, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

### Metadata Based

#### 11. When was the Titanic released?

In [ ]:
query = "When was the Titanic released?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

#### 12. What genre is The Hunger Games?

In [ ]:
query = "What genre is The Hunger Games?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

#### 13. What language is Spirited Away in?

In [ ]:
query = "What language is Spirited Away in?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

### Conceptual-Based

#### 14. Which movie is about a sinking ship and a romance?

In [ ]:
query = "Which movie is about a sinking ship and a romance?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

#### 15. Which movie includes a rebellion against the Capitol?

In [ ]:
query = "Which movie includes a rebellion against the Capitol?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

#### 16. Which movie features the Battle of Hoth?

In [ ]:
query = "Which movie features the Battle of Hoth?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

### Ambiguous-Based

#### 17. tell me about the movie where the ship sinks and the couple falls in love

In [ ]:
query = "tell me about the movie where the ship sinks and the couple falls in love"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

#### 18. the hunger games movie plot

In [ ]:
query = "the hunger games movie plot"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

#### 20. when did that star wars empire movie come out

In [ ]:
query = "when did that star wars empire movie come out"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

### No Retrival Expected

In [ ]:
query = "What is the plot of Avatar 2?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

In [ ]:
query = "What happens in Inception?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

In [ ]:
query = "Tell me about the WWE"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

In [ ]:
query = "What is the plot of the banana spaceship movie?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

In [ ]:
query = "Who is the main character in the invisible dragon film?"
documents, ids = retrieve_documents(query, collection, embedding_model, top_k=3)
print_test_output(documents, ids)

# Answer Generating

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Define the 4-bit configuration
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Loading model in 4-bit to fit Mistral-7B in T4 memory
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

In [ ]:
def generate_answer(query, context, context_ids, model, tokenizer):
    """
    ConcatenAtes relevant movie chunks and asks Mistral to answer the query based on them.
    """
    # Combine the retrieved docs into a single text block
    context_block = "\n\n".join(context)

    # Create the prompt using Mistral's specific instruction format
    prompt = f"""<s>[INST] You are an expert movie assistant. Use the following context to answer the question.
    If you don't know the answer based on the context, just say you don't know. Do not spoil any movies that you mention.

    Context:
    {context}

    Question: {query}
    [/INST]"""

    # Convert prompt to tokens and send to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # Generate the answer
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,          # Temperature is set to 0.7, seems to be decent
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )


    # Calculate the exact length of the input prompt
    input_length = inputs['input_ids'].size(1)

    # Slice the output tensor to ignore the prompt, then decode only the new tokens
    answer_only = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    return answer_only.strip()

# Full RAG Pipeline (Testing)

In [ ]:
def rag_pipeline(query, collection, embedding_model, model, tokenizer, top_k = 3):
  """
  Retrieve the relevant movies based on the query and send that to the model for an answer
  """
  # Retrieve the context documents
  context, context_ids = retrieve_documents(query, collection, embedding_model, top_k = top_k)

  # Print the retrieved documents
  print_test_output(context, context_ids)

  # Generate the answer with the retrieved context
  answer = generate_answer(query, context, context_ids, model, tokenizer)

  return answer

In [ ]:
import textwrap

In [ ]:
query = "What is a good movie about a road trip gone wrong?"
answer = rag_pipeline(query, collection, embedding_model, model, tokenizer, top_k = 5)
print("\nMistral Response:\n")
print(textwrap.fill(answer, width=80))

In [ ]:
query = "I'm craving a good movie about being put into a new environment and having to adjust and eventually thrive?"
answer = rag_pipeline(query, collection, embedding_model, model, tokenizer, top_k = 5)
print("\nMistral Response:\n")
print(textwrap.fill(answer, width=80))

In [ ]:
query = "What are some good movies to put on the TV during a family gathering?"
answer = rag_pipeline(query, collection, embedding_model, model, tokenizer, top_k = 5)
print("\nMistral Response:\n")
print(textwrap.fill(answer, width=80))